# Data Visualizations

In [2]:
# import packages 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pywaffle import Waffle
from matplotlib.ticker import FuncFormatter

In [ ]:
# colors 

'#f5f0e8'   # background
'#1a1a1a'   # body
'#06395b'
'#0c629b'
"#458dbd"
"#adddfc"
'#1a5c3a'
"#22794c"
"#67a384"
"#b4d9c6"
"#c48b22"
"#e3b96b"
"#f7dfb3"
"#b84141"

color1 = ['#06395b', "#458dbd", "#adddfc", 'rgba(6, 57, 91, 0.3)', 'rgba(69, 141, 189, 0.3)' ]
color2 = ['#1a5c3a', "#4f916f", "#b4d9c6", 'rgba(79, 145, 111, 0.3)', 'rgba(180, 217, 198, 0.3)' ]
color3 = ["#c48b22", "#e3b96b", "#f7dfb3", 'rgba(196, 139, 34, 0.3)', 'rgba(227, 185, 107, 0.3)' ]

In [45]:
# theme 
sns.set_theme(
    style="whitegrid",
    rc={
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 20
    }
)

font_manager.fontManager.addfont("PlayfairDisplay-VariableFont_wght.ttf")

## Overall Percent Spending used on Food

Food personal consumption expenditures / total personal consumption expenditures over time = what percent of spending is used for food over time in general


In [6]:
# import data

food_pce = pd.read_csv("../data/total/DFXARC1M027SBEA.csv", low_memory=False)
pce = pd.read_csv("../data/total/PCE.csv", low_memory=False)

In [23]:
# clean data 

# filter to 1960 and later 
food_pce_clean = food_pce.loc[food_pce['observation_date']>='1960-01-01']
pce_clean = pce.loc[pce['observation_date']>='1960-01-01']

# rename 
food_pce_clean = food_pce_clean.rename(columns={'DFXARC1M027SBEA':'food_pce'})
pce_clean = pce_clean.rename(columns={'PCE':'pce'})

# merge 
percent_food_expend = food_pce_clean.merge(pce_clean, how='outer')

# divide to get percentage
percent_food_expend['percent'] = (percent_food_expend['food_pce']/percent_food_expend['pce']) *100

In [91]:
# line chart of food as a percent of personal consumption expenditures

# data 
y = percent_food_expend["percent"]
x = percent_food_expend["observation_date"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="",
        line=dict(
            color='#67a384',
            width=4
        ),
        fill='tozeroy',
        fillcolor='rgba(103, 163, 132, 0.18)',
        hovertemplate=
        "Food Percentage: %{y:.1f}%<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food Expenditure as a Share of Total Personal Consumption Expenditures",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=10
    ),

    yaxis=dict(
        title="Percentage",
        range=[0, 100],
        ticksuffix="%",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    )
)

# endpoint annotation
fig.add_annotation(
    x=x.iloc[-1],
    y=y.iloc[-1],
    text=f"{y.iloc[-1]:.1f}%",
    showarrow=False,
    xshift=20,
    font=dict(
        color="#f7dfb3",
        size=14
    )
)

fig.show()

In [92]:
# save fig 
fig.write_html(f"../website/charts/percent_food_expenditure.html", include_plotlyjs="cdn", full_html=False)

## Food CPI vs General CPI 

Food cpi vs general cpi  = whether food prices have inflated faster or slower than everything else in the economy over time / is food getting more expensive relative to other things Americans buy?

In [121]:
# import data 

food_cpi = pd.read_csv("../data/total/CPIUFDSL.csv", low_memory=False)
general_cpi = pd.read_csv("../data/total/CPIAUCSL.csv", low_memory=False)

In [122]:
# clean data 

# filter to 1960 and later 
food_cpi_clean = food_cpi.loc[food_cpi['observation_date']>='1960-01-01']
general_cpi_clean = general_cpi.loc[general_cpi['observation_date']>='1960-01-01']

# rename 
food_cpi_clean = food_cpi_clean.rename(columns={'CPIUFDSL':'food_cpi'})
general_cpi_clean = general_cpi_clean.rename(columns={'CPIAUCSL':'total_cpi'})

# merge 
both_cpi = food_cpi_clean.merge(general_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_food = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_cpi"].values
base_all = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "total_cpi"].values

both_cpi['food_cpi_reindex'] = (both_cpi['food_cpi'] / base_food) * 100
both_cpi['total_cpi_reindex'] = (both_cpi['total_cpi'] / base_all) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [132]:
# line chart of food as a percent of personal consumption expenditures

# data 
x = both_cpi["observation_date"]
food_cpi = both_cpi["food_cpi_reindex"]
total_cpi = both_cpi["total_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        hovertemplate=
        "All Items CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_cpi,
        mode="lines",
        name="Food",
        line=dict(
            color='#67a384',
            width=4
        ),
        hovertemplate=
        "Food CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food vs. Overall Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [133]:
# save fig 
fig.write_html(f"../website/charts/food_vs_overall_inflation.html", include_plotlyjs="cdn", full_html=False)

## Food at Home vs Food Away from Home

Food at home vs food away from home CPIs over time = whether eating at home vs eating out was cheaper at certain times 

In [135]:
# import data 

food_home_cpi = pd.read_csv("../data/categories/CUUR0000SAF11.csv", low_memory=False)
food_away_cpi = pd.read_csv("../data/categories/CUUR0000SEFV.csv", low_memory=False)

In [136]:
# clean data 

# filter to 1960 and later 
food_home_cpi_clean = food_home_cpi.loc[food_home_cpi['observation_date']>='1960-01-01']
food_away_cpi_clean = food_away_cpi.loc[food_away_cpi['observation_date']>='1960-01-01']

# rename 
food_home_cpi_clean = food_home_cpi_clean.rename(columns={'CUUR0000SAF11':'food_home_cpi'})
food_away_cpi_clean = food_away_cpi_clean.rename(columns={'CUUR0000SEFV':'food_away_cpi'})

# merge 
both_cpi = food_home_cpi_clean.merge(food_away_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_home = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_home_cpi"].values
base_away = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_away_cpi"].values

both_cpi['food_home_cpi_reindex'] = (both_cpi['food_home_cpi'] / base_home) * 100
both_cpi['food_away_cpi_reindex'] = (both_cpi['food_away_cpi'] / base_away) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [139]:
# line chart of food at home vs food away from home cpis 

# data 
x = both_cpi["observation_date"]
food_home_cpi = both_cpi["food_home_cpi_reindex"]
food_away_cpi = both_cpi["food_away_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Food at Home CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_away_cpi,
        mode="lines",
        name="Food Away from Home",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Food Away from Home CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food at Home vs. Away from Home Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [140]:
# save fig 
fig.write_html(f"../website/charts/home_vs_away.html", include_plotlyjs="cdn", full_html=False)

## Grocery Category Comparison 

Grocery categories cpi comparison = relative price trajectory of different food types, is cheaper food less healthy now


In [150]:
# import data 

cereal_bakery = pd.read_csv("../data/categories/CUUR0000SAF111.csv", low_memory=False)
meat_fish_eggs = pd.read_csv("../data/categories/CUUR0000SAF112.csv", low_memory=False)
fruit_veg = pd.read_csv("../data/categories/CUUR0000SAF113.csv", low_memory=False)
beverage = pd.read_csv("../data/categories/CUUR0000SAF114.csv", low_memory=False)

# clean data 

# filter to 1960 and later 
cereal_bakery = cereal_bakery.loc[cereal_bakery['observation_date']>='1967-01-01']
meat_fish_eggs = meat_fish_eggs.loc[meat_fish_eggs['observation_date']>='1967-01-01']
fruit_veg = fruit_veg.loc[fruit_veg['observation_date']>='1967-01-01']
beverage = beverage.loc[beverage['observation_date']>='1967-01-01']

# rename 
cereal_bakery = cereal_bakery.rename(columns={'CUUR0000SAF111':'cereal_bakery_cpi'})
meat_fish_eggs = meat_fish_eggs.rename(columns={'CUUR0000SAF112':'meat_fish_eggs_cpi'})
fruit_veg = fruit_veg.rename(columns={'CUUR0000SAF113':'fruit_veg_cpi'})
beverage = beverage.rename(columns={'CUUR0000SAF114':'beverage_cpi'})

# reindex to January 1967 = 100
base_cb = cereal_bakery.loc[cereal_bakery['observation_date']=="1967-01-01", "cereal_bakery_cpi"].values
base_mfe = meat_fish_eggs.loc[meat_fish_eggs['observation_date']=="1967-01-01", "meat_fish_eggs_cpi"].values
base_fv = fruit_veg.loc[fruit_veg['observation_date']=="1967-01-01", "fruit_veg_cpi"].values
base_b = beverage.loc[beverage['observation_date']=="1967-01-01", "beverage_cpi"].values

cereal_bakery['cereal_bakery_cpi_reindex'] = (cereal_bakery['cereal_bakery_cpi'] / base_cb) * 100
meat_fish_eggs['meat_fish_eggs_cpi_reindex'] = (meat_fish_eggs['meat_fish_eggs_cpi'] / base_mfe) * 100
fruit_veg['fruit_veg_cpi_reindex'] = (fruit_veg['fruit_veg_cpi'] / base_fv) * 100
beverage['beverage_cpi_reindex'] = (beverage['beverage_cpi'] / base_b) * 100

# drop na 
cereal_bakery = cereal_bakery.dropna()
meat_fish_eggs = meat_fish_eggs.dropna()
fruit_veg = fruit_veg.dropna()
beverage = beverage.dropna()

In [156]:
# line chart of grocery category cpis 

# data 
x = cereal_bakery["observation_date"]
cb = cereal_bakery["cereal_bakery_cpi_reindex"]
mfe = meat_fish_eggs["meat_fish_eggs_cpi_reindex"]
fv = fruit_veg["fruit_veg_cpi_reindex"]
b = beverage["beverage_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=cb,
        mode="lines",
        name="Cereal and Bakery",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Cereal and Bakery CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=mfe,
        mode="lines",
        name="Meat, Poultry, Fish, Eggs",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Meat, Poultry, Fish, Eggs CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=fv,
        mode="lines",
        name="Fruit and Vegetables",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Fruit and Vegetables CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=b,
        mode="lines",
        name="Beverages",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Beverages CPI: %{y:.1f}<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Grocery Price Inflation by Category",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1
    )
)

fig.show()

In [157]:
# save fig 
fig.write_html(f"../website/charts/grocery_categories.html", include_plotlyjs="cdn", full_html=False)

## Animated Bar Chart by Outlet 

Monthly sales by outlet = compare sales by outlet over time in order to see whether more people are food at grocery stores, large stores like costco, restaurants, etc

In [158]:
# import data 

sales_by_outlet = pd.read_csv("../data/usda/monthly_sales_by_outlet.csv", low_memory=False)

In [160]:
sales_by_outlet.columns

Index(['Year', 'Month',
       'Food-at-home sales at grocery stores (millions of nominal U.S. dollars)',
       'Food-at-home sales at warehouse clubs and supercenters (millions of nominal U.S. dollars)',
       'Other food-at-home sales (millions of nominal U.S. dollars)',
       'Total food-at-home sales (millions of nominal U.S. dollars)',
       'Food sales at full-service restaurants (millions of nominal U.S. dollars)',
       'Food sales at limited-service restaurants (millions of nominal U.S. dollars)',
       'Other food-away-from-home sales (millions of nominal U.S. dollars)',
       'Total food-away-from-home sales (millions of nominal U.S. dollars)',
       'Total food sales (millions of nominal U.S. dollars)',
       'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))',
       'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))',
       'Other food-at-home sales (millions of constant U.S. dollars (1988=100))

In [ ]:
# clean data 

# select columns 
sales_by_outlet = sales_by_outlet[['Year',
                                   'Month',
                                   'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))',
                                   'Other food-at-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))',
                                   'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food-at-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food-away-from-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food sales (millions of constant U.S. dollars (1988=100))']]

# rename columns 
sales_by_outlet = sales_by_outlet.rename({
    'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))': 'grocery_stores',
    'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))': 'warehouse_supercenter',
    'Other food-at-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_at_home',
    'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))': 'full_restaurants',
    'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))': 'limited_restaurants',
    'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_away',
    'Total food-at-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_at_home',
    'Total food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_away',
    'Total food sales (millions of constant U.S. dollars (1988=100))': 'total_overall'
})

# 

In [159]:
sales_by_outlet

,Year,Month,Food-at-home sales at grocery stores (millions of nominal U.S. dollars),Food-at-home sales at warehouse clubs and supercenters (millions of nominal U.S. dollars),Other food-at-home sales (millions of nominal U.S. dollars),Total food-at-home sales (millions of nominal U.S. dollars),Food sales at full-service restaurants (millions of nominal U.S. dollars),Food sales at limited-service restaurants (millions of nominal U.S. dollars),Other food-away-from-home sales (millions of nominal U.S. dollars),Total food-away-from-home sales (millions of nominal U.S. dollars),Total food sales (millions of nominal U.S. dollars),Food sales at grocery stores (millions of constant U.S. dollars (1988=100)),Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100)),Other food-at-home sales (millions of constant U.S. dollars (1988=100)),Total food-at-home sales (millions of constant U.S. dollars (1988=100)),Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100)),Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100)),Other food-away-from-home sales (millions of constant U.S. dollars (1988=100)),Total food-away-from-home sales (millions of constant U.S. dollars (1988=100)),Total food sales (millions of constant U.S. dollars (1988=100))
0,2024,December,"53,508.70","27,638.43","17,470.23","98,617.36","49,109.64","44,980.43","22,796.67","116,886.74","215,504.10","20,233.37","10,452.11","6,606.61","37,292.09","15,968.79","14,623.75","7,411.56","38,004.10","75,296.19"
1,2024,November,"51,907.11","24,328.66","15,512.88","91,748.65","46,314.84","44,311.91","22,347.16","112,973.91","204,722.56","19,653.77","9,212.64","5,874.26","34,740.67","15,104.93","14,449.37","7,287.15","36,841.45","71,582.12"
2,2024,October,"51,349.68","23,279.29","15,250.78","89,879.75","46,335.73","46,825.90","22,793.14","115,954.77","205,834.52","19,428.32","8,808.75","5,770.76","34,007.83","15,154.10","15,311.94","7,453.47","37,919.51","71,927.34"
3,2024,September,"49,193.49","21,906.63","13,987.02","85,087.14","44,877.04","44,574.77","22,187.63","111,639.44","196,726.58","18,638.70","8,301.00","5,300.03","32,239.73","14,711.87","14,610.42","7,272.67","36,594.96","68,834.69"
4,2024,August,"51,375.86","24,113.02","14,548.53","90,037.41","48,088.59","48,359.83","23,034.21","119,482.63","209,520.04","19,552.29","9,177.77","5,537.34","34,267.40","15,818.17","15,904.82","7,575.77","39,298.76","73,566.16"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,1997,May,"23,432.74","2,149.88","5,610.42","31,193.04","10,509.64","10,112.04","4,447.09","25,068.77","56,261.81","17,352.94","1,589.65","4,158.24","23,100.83","8,191.35","7,881.17","3,466.22","19,538.74","42,639.57"
332,1997,April,"21,509.06","1,914.52","5,403.54","28,827.12","9,831.41","9,332.29","4,205.16","23,368.86","52,195.98","15,928.37","1,416.12","4,004.20","21,348.69","7,667.63","7,278.10","3,279.75","18,225.48","39,574.17"
333,1997,March,"22,756.94","1,969.54","5,613.02","30,339.50","10,093.29","9,340.53","4,159.77","23,593.59","53,933.09","16,831.10","1,446.53","4,162.58","22,440.21","7,881.97","7,293.87","3,248.51","18,424.35","40,864.56"
334,1997,February,"20,104.82","1,694.26","5,246.76","27,045.84","9,094.30","8,255.88","3,922.10","21,272.28","48,318.12","14,869.59","1,259.08","3,875.43","20,004.10","7,120.10","6,463.46","3,070.77","16,654.33","36,658.43"


In [162]:
df

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4
...,...,...,...,...,...,...,...,...
1699,Zimbabwe,Africa,1987,62.351,9216418,706.157306,ZWE,716
1700,Zimbabwe,Africa,1992,60.377,10704340,693.420786,ZWE,716
1701,Zimbabwe,Africa,1997,46.809,11404948,792.449960,ZWE,716
1702,Zimbabwe,Africa,2002,39.989,11926563,672.038623,ZWE,716


In [161]:
import plotly.express as px

df = px.data.gapminder()

fig = px.bar(df, x="continent", y="pop", color="continent",
  animation_frame="year", animation_group="country", range_y=[0,4000000000])
fig.show()